# Coletar 1 paper com pelo menos 3 reviews do OpenReview

Este notebook procura um paper em um venue do OpenReview que tenha pelo menos 3 reviews oficiais públicas/acessíveis para sua conta.

Ele salva:

- dados do paper;
- URL do paper;
- URL/PDF quando disponível;
- textos das reviews;
- rating, confidence e recommendation quando disponíveis.

Se aparecer `ChallengeRequiredError`, abra o link mostrado, complete a verificação e rode a célula novamente.

In [ ]:
!pip -q install openreview-py pandas

## Autenticação

No Colab, você pode usar Secrets com estes nomes:

- `OPENREVIEW_EMAIL`
- `OPENREVIEW_PASSWORD`

Se os Secrets não estiverem configurados, o notebook pede email e senha.

In [ ]:
import getpass
from pprint import pprint

import openreview
import pandas as pd
from openreview import OpenReviewException


def get_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None


OPENREVIEW_EMAIL = get_colab_secret("OPENREVIEW_EMAIL")
OPENREVIEW_PASSWORD = get_colab_secret("OPENREVIEW_PASSWORD")

if not OPENREVIEW_EMAIL:
    OPENREVIEW_EMAIL = input("OpenReview email: ").strip()

if not OPENREVIEW_PASSWORD:
    OPENREVIEW_PASSWORD = getpass.getpass("OpenReview password: ")

client = openreview.api.OpenReviewClient(
    baseurl="https://api2.openreview.net",
    username=OPENREVIEW_EMAIL,
    password=OPENREVIEW_PASSWORD,
    tokenExpiresIn=60 * 60 * 24,
)

print("Cliente OpenReview criado.")

## Configuração

Troque o `VENUE_ID` se quiser outro evento.

Exemplos:

- `ICLR.cc/2024/Conference`
- `ICLR.cc/2025/Conference`
- `NeurIPS.cc/2024/Conference`

`MAX_PAPERS_TO_SCAN` controla quantos papers o notebook tenta verificar até encontrar um com pelo menos 3 reviews.

In [ ]:
VENUE_ID = "ICLR.cc/2024/Conference"
MIN_REVIEWS = 3
MAX_PAPERS_TO_SCAN = 100
BATCH_SIZE = 25

SUBMISSION_INVITATION = f"{VENUE_ID}/-/Submission"
REVIEW_TYPE = "Official_Review"

print("Venue:", VENUE_ID)
print("Submission invitation:", SUBMISSION_INVITATION)
print("Mínimo de reviews:", MIN_REVIEWS)

## Funções auxiliares

In [ ]:
def content_value(note_or_dict, key, default=None):
    content = note_or_dict.content if hasattr(note_or_dict, "content") else note_or_dict.get("content", {})
    value = content.get(key, default)
    if isinstance(value, dict) and "value" in value:
        return value["value"]
    return value


def note_id(note_or_dict):
    return note_or_dict.id if hasattr(note_or_dict, "id") else note_or_dict.get("id")


def note_number(note_or_dict):
    return note_or_dict.number if hasattr(note_or_dict, "number") else note_or_dict.get("number")


def note_forum(note_or_dict):
    return note_or_dict.forum if hasattr(note_or_dict, "forum") else note_or_dict.get("forum")


def note_invitations(note_or_dict):
    return note_or_dict.invitations if hasattr(note_or_dict, "invitations") else note_or_dict.get("invitations", [])


def note_content(note_or_dict):
    return note_or_dict.content if hasattr(note_or_dict, "content") else note_or_dict.get("content", {})


def get_replies(submission):
    details = submission.details if hasattr(submission, "details") else submission.get("details", {})
    return details.get("replies", [])


def is_official_review(reply, venue_id, submission_number):
    invitations = note_invitations(reply)
    expected_piece = f"{venue_id}/Submission{submission_number}/-/{REVIEW_TYPE}"
    if any(expected_piece in invitation for invitation in invitations):
        return True

    invitation_text = " ".join(invitations).lower()
    return "official_review" in invitation_text or "official-review" in invitation_text


def extract_text_fields(note_or_dict):
    fields = [
        "review",
        "summary",
        "main_review",
        "strengths",
        "weaknesses",
        "strengths_and_weaknesses",
        "limitations",
        "questions",
        "comment",
    ]
    parts = []
    for field in fields:
        value = content_value(note_or_dict, field, None)
        if value:
            parts.append(f"[{field}]\n{value}")
    return "\n\n".join(parts) if parts else None


def print_openreview_error(error):
    print("Erro OpenReview:")
    print(error)
    challenge_url = error.get("details", {}).get("challengeUrl") if isinstance(error, dict) else None
    if challenge_url:
        print("\nAbra este link, complete a verificação e rode a célula novamente:")
        print(challenge_url)

## Procurar um paper com pelo menos 3 reviews

Esta célula varre submissões em lotes até encontrar um paper com `MIN_REVIEWS` reviews oficiais.

In [ ]:
selected_submission = None
selected_reviews = []
scanned = 0

try:
    for offset in range(0, MAX_PAPERS_TO_SCAN, BATCH_SIZE):
        submissions = client.get_notes(
            invitation=SUBMISSION_INVITATION,
            details="replies",
            limit=BATCH_SIZE,
            offset=offset,
        )
        if not submissions:
            break

        for submission in submissions:
            scanned += 1
            submission_number = note_number(submission)
            replies = get_replies(submission)
            reviews = [
                reply for reply in replies
                if is_official_review(reply, VENUE_ID, submission_number)
            ]

            title = content_value(submission, "title", "Sem título")
            print(f"#{submission_number}: {len(reviews)} reviews — {title[:90]}")

            if len(reviews) >= MIN_REVIEWS:
                selected_submission = submission
                selected_reviews = reviews
                break

        if selected_submission is not None:
            break

except OpenReviewException as e:
    error = e.args[0] if e.args else {}
    print_openreview_error(error)

print("\nPapers verificados:", scanned)
if selected_submission is None:
    print(f"Nenhum paper com pelo menos {MIN_REVIEWS} reviews encontrado no intervalo verificado.")
else:
    print("\nPaper selecionado:")
    print("Número:", note_number(selected_submission))
    print("Título:", content_value(selected_submission, "title", ""))
    print("Reviews:", len(selected_reviews))

## Montar tabela do paper selecionado

In [ ]:
rows = []

if selected_submission is None:
    df = pd.DataFrame()
    print("Nenhum paper selecionado.")
else:
    submission = selected_submission
    submission_id = note_id(submission)
    submission_number = note_number(submission)
    forum = note_forum(submission) or submission_id

    title = content_value(submission, "title", "")
    abstract = content_value(submission, "abstract", "")
    authors = content_value(submission, "authors", [])
    venue = content_value(submission, "venue", "")
    pdf_url = content_value(submission, "pdf", "")
    if pdf_url and isinstance(pdf_url, str) and pdf_url.startswith("/"):
        pdf_url = "https://openreview.net" + pdf_url

    paper_url = f"https://openreview.net/forum?id={forum}"

    for idx, review in enumerate(selected_reviews, start=1):
        rows.append({
            "paper_number": submission_number,
            "paper_id": submission_id,
            "paper_url": paper_url,
            "pdf_url": pdf_url,
            "title": title,
            "abstract": abstract,
            "authors": authors,
            "venue": venue,
            "review_index": idx,
            "review_id": note_id(review),
            "review_text": extract_text_fields(review),
            "rating": content_value(review, "rating", None),
            "confidence": content_value(review, "confidence", None),
            "recommendation": content_value(review, "recommendation", None),
        })

    df = pd.DataFrame(rows)
    print("Linhas/reviews:", len(df))

df

## Visualizar paper e reviews

In [ ]:
if df.empty:
    print("Nenhum dado para visualizar.")
else:
    first = df.iloc[0]
    print("=" * 100)
    print(f"Paper #{first['paper_number']}: {first['title']}")
    print(f"URL: {first['paper_url']}")
    print(f"PDF: {first['pdf_url']}")
    print("\nAbstract:")
    print(first["abstract"])
    print("\nReviews:")

    for _, row in df.iterrows():
        print("-" * 80)
        print(f"Review #{row['review_index']} | ID: {row['review_id']}")
        print(f"Rating: {row['rating']}")
        print(f"Confidence: {row['confidence']}")
        print(f"Recommendation: {row['recommendation']}")
        print("\nTexto:")
        print(row["review_text"])

## Salvar CSV e JSON

In [ ]:
csv_path = "openreview_one_paper_3_reviews.csv"
json_path = "openreview_one_paper_3_reviews.json"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print("Arquivos salvos:")
print("-", csv_path)
print("-", json_path)

## Baixar arquivos no Colab

In [ ]:
try:
    from google.colab import files
    files.download(csv_path)
    files.download(json_path)
except Exception as e:
    print("Download automático disponível apenas no Google Colab.")
    print(e)

## Debug: inspecionar campos de uma review

Use isto se `review_text` vier vazio. Alguns venues usam nomes de campo diferentes.

In [ ]:
if not selected_reviews:
    print("Nenhuma review selecionada para inspecionar.")
else:
    sample_review = selected_reviews[0]
    pprint(note_content(sample_review))
    print("\nInvitations:")
    pprint(note_invitations(sample_review))